In [1]:
import pandas as pd

In [2]:
df=pd.read_csv("12._IMDB Dataset.csv")

In [3]:
df.drop_duplicates(inplace=True)

#Pre_Processing
#1.Converting to The lower case

In [4]:
df["review"]=df["review"].str.lower()

2.Removinf the urls

In [5]:
import re

def remove_urls(text):#Text represent each review
    text=re.sub(r"http\S+","",text) #(Pattern,repl,string)
    return text
df["review"]=df["review"].apply(remove_urls)



3.Removing Punctuations

In [6]:
def remove_punc(text):
    text=re.sub(r"[^A-Za-z0-9\s]","",text)#only kept A-Za_z0_9 and remove other things
    return text

df["review"]=df["review"].apply(remove_punc)

4.Removing HTML

In [7]:
def remove_html(text):
    text=re.sub(r"<.*?>","",text)
    return text

df["review"]=df["review"].apply(remove_html)

5.Removing The Stopwords

In [8]:
import nltk

nltk.download("punkt") #For Tokenization
nltk.download("punkt_tab")
nltk.download("stopwords")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Dell\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Dell\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Dell\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [9]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [10]:
def remove_stopwords(text):
    tokens=word_tokenize(text)
    stop_words=stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text=text.replace(word,"")
    return text

df["review"]=df["review"].apply(remove_stopwords)

In [12]:
df.head(5)

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


6.stemming

In [13]:
#Running->run
#played->play

from nltk.stem import PorterStemmer

In [14]:
def Stemming(text):
    ps=PorterStemmer()
    stemmed_words=[]

    tokens=word_tokenize(text)
    for token in tokens:
        stemmed_token=ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

In [15]:
df.head(2)

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive


7.Encoding

In [16]:
from sklearn.preprocessing import LabelEncoder

le=LabelEncoder()
df["sentiment"]=le.fit_transform(df["sentiment"])

y=df["sentiment"]

8.Vectorization

In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf=TfidfVectorizer(max_features=5000)
x=tf.fit_transform(df["review"])


9.Datast and DataLoader

In [19]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test=train_test_split(
    x,y,test_size=0.2,random_state=42
)

In [20]:
import torch
from torch.utils.data import TensorDataset,DataLoader

In [23]:
x_train=x_train.toarray()
x_test=x_test.toarray()


In [24]:
train_set=TensorDataset(
    torch.from_numpy(x_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set=TensorDataset(
    torch.from_numpy(x_test).float(),
    torch.from_numpy(y_test.values).float()
)


In [25]:
train_loader=DataLoader(train_set,shuffle=True,batch_size=64)
test_loader=DataLoader(test_set,shuffle=True,batch_size=64)


In [26]:
import torch.nn as nn
import torch.optim as optim

In [27]:
class RNN(nn.Module):
    def __init__(self,input_size,hidden_size=128,num_layers=1):
        super().__init__()
        self.hidden_size=hidden_size
        self.num_layers=num_layers

        #RNN Layer
        self.rnn=nn.RNN(input_size,hidden_size,num_layers,batch_first=True)

        #Fully Connected Layer
        self.fc=nn.Linear(hidden_size,1)

    def forward(self,x):
        h0=torch.zeros(self.num_layers,x.size(0),self.hidden_size)

        out,_=self.rnn(x,h0)
        # 1st value=hidden state of all the timestamps=>(batch,seq_len,hidden size)
        #2nd value=fianl hidden state of last timestep

        out=self.fc(out[:,-1,:])
        return out
        

In [28]:
input_size=x_train.shape[1]

model=RNN(input_size)
criterion=nn.BCELoss()
optimizer=optim.Adam(model.parameters())


In [29]:
epochs=10
for epoch in range(epochs):
    model.train()

    for xb,yb in train_loader:
        optimizer.zero_grad()

        xb=xb.unsqueeze(1) #adding singlton direction, 2D->3D

        outputs=model(xb)

        outputs=torch.sigmoid(outputs.squeeze())

        loss=criterion(outputs,yb)
        loss.backward()
        optimizer.step()

    print(f"epoch={epoch+1}/{epochs} and loss ={loss.item()}")


    

epoch=1/10 and loss =0.26300138235092163
epoch=2/10 and loss =0.2429625391960144
epoch=3/10 and loss =0.4894869029521942
epoch=4/10 and loss =0.2373717725276947
epoch=5/10 and loss =0.13199524581432343
epoch=6/10 and loss =0.1760091930627823
epoch=7/10 and loss =0.18712890148162842
epoch=8/10 and loss =0.2239171415567398
epoch=9/10 and loss =0.37954583764076233
epoch=10/10 and loss =0.22899818420410156


In [30]:
model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0
    
    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 85.78199052132702
